# EXP-008 | Target Encoding

시술 유형, 배란 유도 유형 등 고카디널리티 범주형에 Fold-wise Target Encoding 적용.
Data Leakage 방지: train은 OOF 방식, test는 전체 train 기준 인코딩.

| 항목 | 내용 |
|---|---|
| 기반 실험 | EXP-003 피처 |
| 모델 | LightGBM |
| CV | Stratified 5-Fold |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, f1_score, recall_score, precision_score, accuracy_score
import lightgbm as lgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
OUT_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
SEED     = 42
N_FOLDS  = 5
EXP_NO   = 8
AUTHOR   = '조여진'
MODEL_NAME = 'LightGBM'
CV_STR   = f'Stratified {N_FOLDS}-Fold'

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')

## 1. Target Encoding 대상 컬럼

LabelEncoding을 Target Encoding으로 대체. Fold-wise OOF 방식으로 Data Leakage 방지.

| 컬럼 | 기존 처리 | 변경 |
|---|---|---|
| `시술 유형` | LabelEncoding | Target Encoding |
| `특정 시술 유형` | LabelEncoding | Target Encoding |
| `배란 유도 유형` | LabelEncoding | Target Encoding |
| `시술 시기 코드` | LabelEncoding | Target Encoding |
| `난자 출처` | LabelEncoding | Target Encoding |
| `정자 출처` | LabelEncoding | Target Encoding |

In [ ]:
# Target Encoding 대상 (src/preprocessing.py의 LABEL_COLS)
TE_COLS = ['시술 시기 코드', '시술 유형', '특정 시술 유형',
           '배란 유도 유형', '난자 출처', '정자 출처']
GLOBAL_MEAN_SMOOTHING = 20  # 스무딩 강도 (작은 그룹 과적합 방지)

def target_encode_oof(train_df, test_df, cols, target, n_folds=5, smoothing=20, seed=42):
    """Fold-wise Target Encoding (Data Leakage 없음)."""
    train_te = train_df.copy()
    test_te  = test_df.copy()
    global_mean = train_df[target].mean()

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    for col in cols:
        oof_enc = np.zeros(len(train_df))

        for tr_idx, val_idx in skf.split(train_df, train_df[target]):
            tr_fold  = train_df.iloc[tr_idx]
            val_fold = train_df.iloc[val_idx]

            # 스무딩 target encoding: (count * mean + smoothing * global_mean) / (count + smoothing)
            stats = tr_fold.groupby(col)[target].agg(['mean', 'count'])
            stats['smoothed'] = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)

            oof_enc[val_idx] = val_fold[col].map(stats['smoothed']).fillna(global_mean).values

        train_te[col] = oof_enc

        # test: 전체 train 기준으로 인코딩 (leakage 없음 — test에는 target 없음)
        stats_full = train_df.groupby(col)[target].agg(['mean', 'count'])
        stats_full['smoothed'] = (stats_full['count'] * stats_full['mean'] + smoothing * global_mean) / (stats_full['count'] + smoothing)
        test_te[col] = test_df[col].map(stats_full['smoothed']).fillna(global_mean)

    return train_te, test_te

print('Target Encoding 함수 정의 완료')
print(f'대상 컬럼: {TE_COLS}')

## 2. 피처 준비

In [ ]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']   = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부'] = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

# 1단계: pre-encode 피처 (기증 여부) 생성
train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)

# 2단계: Target Encoding (preprocess 전 — 원본 문자열 필요)
train_te, test_te = target_encode_oof(
    train_fe, test_fe, TE_COLS, TARGET,
    n_folds=N_FOLDS, smoothing=GLOBAL_MEAN_SMOOTHING, seed=SEED
)

# 3단계: 공통 전처리 (TE_COLS는 이미 수치형으로 변환됐으므로 LABEL_COLS에서 중복 처리됨)
X_train, X_test = preprocess(train_te, test_te)

# 4단계: 수치형 파생 피처
X_train = add_derived_features(X_train)
X_test  = add_derived_features(X_test)
y_train = train[TARGET]

print(f'X_train: {X_train.shape}  /  X_test: {X_test.shape}')
print(f'결측 컬럼 수: {X_train.isnull().any().sum()}개')

## 3. 모델 학습

In [ ]:
LGB_PARAMS = dict(
    objective='binary', metric='auc', verbosity=-1, seed=SEED,
    is_unbalance=True, learning_rate=0.05, num_leaves=127,
    min_child_samples=50, feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.1, lambda_l2=0.1,
)

skf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_preds  = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))
fold_aucs  = []
models     = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    ds_tr  = lgb.Dataset(X_tr, label=y_tr)
    ds_val = lgb.Dataset(X_val, label=y_val, reference=ds_tr)
    model  = lgb.train(LGB_PARAMS, ds_tr, num_boost_round=2000,
                       valid_sets=[ds_val],
                       callbacks=[lgb.early_stopping(100, verbose=False),
                                  lgb.log_evaluation(period=-1)])
    val_prob = model.predict(X_val)
    auc = roc_auc_score(y_val, val_prob)
    fold_aucs.append(auc)
    oof_preds[val_idx]  = val_prob
    test_preds         += model.predict(X_test) / N_FOLDS
    models.append(model)
    print(f'  Fold {fold}  best_iter={model.best_iteration:4d}  AUC={auc:.5f}')

oof_auc  = roc_auc_score(y_train, oof_preds)
oof_prauc = average_precision_score(y_train, oof_preds)
oof_f1    = f1_score(y_train, (oof_preds >= 0.5).astype(int))
print(f'\nOOF ROC-AUC : {oof_auc:.5f}')
print(f'OOF PR-AUC  : {oof_prauc:.5f}')
print(f'OOF F1      : {oof_f1:.5f}  (threshold=0.5)')
print(f'EXP-003 대비 : {oof_auc - 0.73887:+.5f}')

## 4. Submission 저장 & 실험 기록

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
sub['probability'] = test_preds
auc_str   = f'{oof_auc:.4f}'.replace('.', '')
out_fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_{auc_str}.csv'
sub.to_csv(OUT_DIR / out_fname, index=False)
print(f'저장: {OUT_DIR / out_fname}')

In [ ]:
def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    from openpyxl import load_workbook
    from openpyxl.styles import Font, Alignment, Border, Side, PatternFill
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

oof_binary = (oof_preds >= 0.5).astype(int)
params_str = f'num_leaves=127, lr=0.05, TE_cols={len(TE_COLS)}, smoothing={GLOBAL_MEAN_SMOOTHING}'

log_to_leaderboard(EXP_NO, AUTHOR, MODEL_NAME, params_str,
                   f1_score(y_train,oof_binary), recall_score(y_train,oof_binary),
                   precision_score(y_train,oof_binary), accuracy_score(y_train,oof_binary),
                   oof_auc, CV_STR, 'v1+TE', X_train.shape[1], 'is_unbalance=True',
                   'N', None, 'notebooks/07_target_enc_yjcho.ipynb',
                   f'Target Encoding {TE_COLS}, smoothing={GLOBAL_MEAN_SMOOTHING}', '')